# Movie Data Processing and Narrative Generation
This notebook loads movie data from multiple streaming platform datasets, removes duplicates, and generates a natural language narrative for each movie.

In [ ]:
import pandas as pd
import os

# Load datasets
datasets_dir = 'Datasets'
files = ['amazon_prime_titles.csv', 'disney_plus_titles.csv', 'hulu_titles.csv', 'netflix_titles.csv']

dfs = []
for file in files:
    path = os.path.join(datasets_dir, file)
    if os.path.exists(path):
        df = pd.read_csv(path)
        df['source'] = file.split('_')[0]
        dfs.append(df)

# Concatenate all datasets
all_data = pd.concat(dfs, ignore_index=True)
print(f"Total records: {len(all_data)}")

In [ ]:
# Filter for movies only
movies = all_data[all_data['type'] == 'Movie'].copy()
print(f"Total movies: {len(movies)}")

In [ ]:
# Drop duplicated movies based on title
# Some titles might have different casing or trailing spaces, so let's clean them first
movies['title_clean'] = movies['title'].astype(str).str.strip().str.lower()
movies = movies.drop_duplicates(subset=['title_clean'], keep='first')
print(f"Total movies after dropping duplicates: {len(movies)}")

In [ ]:
# Fill missing values with empty strings for easier string formatting
fill_cols = ['director', 'listed_in', 'description', 'country', 'cast']
movies[fill_cols] = movies[fill_cols].fillna('')

def generate_narrative(row):
    title = row['title']
    genre = row['listed_in'] if row['listed_in'] else "genre-defying"
    director = row['director']
    year = row['release_year']
    desc = row['description']
    
    # Constructing the narrative using f-strings
    if director:
        narrative = f"'{title}' is a {genre} film directed by {director}, released in {year}. The plot follows: {desc}"
    else:
        narrative = f"'{title}' is a {genre} film released in {year}. The plot follows: {desc}"
        
    return narrative

movies['content_narrative'] = movies.apply(generate_narrative, axis=1)

In [ ]:
# Show some of the generated narratives
for idx, narrative in enumerate(movies['content_narrative'].head(5)):
    print(f"Narrative {idx + 1}:")
    print(narrative)
    print("-" * 50)